<a href="https://colab.research.google.com/github/msfasha/307401-Big-Data/blob/main/20252/Module%205%20-%20Introduction%20to%20Apache%20Spark/4-Machine_Learning_with_MLlib.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color:#E25822; font-family:Arial, sans-serif; font-size:2.4em; border-bottom: 3px solid #E25822; padding-bottom:10px;">
  Machine Learning with Apache Spark MLlib
</h1>

<p style="font-size:1.1em; color:#444;">
  In this notebook we explore <strong>Apache Spark MLlib</strong> — Spark's scalable machine learning library.
  We cover <em>regression</em>, <em>classification</em>, <em>clustering</em>, <em>ML Pipelines</em>, and
  <em>hyperparameter tuning</em> using real-world inspired datasets generated entirely inline.
</p>

| Topic | Algorithms |
|---|---|
| Regression | Linear Regression, Decision Tree Regressor |
| Classification | Logistic Regression, Random Forest Classifier |
| Clustering | K-Means |
| Pipelines | StringIndexer → OneHotEncoder → VectorAssembler → StandardScaler → RF |
| Tuning | ParamGridBuilder, CrossValidator |

<h2 style="color:#E25822; font-family:Arial, sans-serif;">1. MLlib Overview</h2>

### What is MLlib?

**Apache Spark MLlib** is Spark's built-in machine learning library. It provides:

- **Scalability**: runs across hundreds of nodes on datasets that do not fit in memory on a single machine.
- **Unified API**: the same code works locally (for development) and on a cluster (for production).
- **DataFrame-based API** (`spark.ml`): the modern, recommended API built on DataFrames and Pipelines.
- **RDD-based API** (`spark.mllib`): the legacy API, still available but not actively developed.

### Supported Algorithm Families

| Family | Examples |
|---|---|
| **Regression** | Linear Regression, Decision Tree, Random Forest, Gradient Boosted Trees, Isotonic Regression |
| **Classification** | Logistic Regression, Decision Tree, Random Forest, GBT, Naïve Bayes, Linear SVM, MLP |
| **Clustering** | K-Means, Bisecting K-Means, Gaussian Mixture, LDA |
| **Recommendation** | ALS (Alternating Least Squares) |
| **Dimensionality Reduction** | PCA, SVD |
| **Feature Engineering** | VectorAssembler, StandardScaler, MinMaxScaler, StringIndexer, OneHotEncoder, Tokenizer, TF-IDF |

### The Pipeline Concept

A **Pipeline** chains multiple `Transformers` and `Estimators` into a single reusable workflow:

```
Raw Data
   │
   ▼
┌─────────────────┐
│  StringIndexer  │  ← Transformer: converts string labels to numeric indices
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│ OneHotEncoder   │  ← Transformer: converts indices to binary vectors
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│ VectorAssembler │  ← Transformer: merges columns into a single feature vector
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│ StandardScaler  │  ← Estimator/Transformer: learns mean & std, then scales
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│   ML Algorithm  │  ← Estimator: learns model parameters from training data
└────────┬────────┘
         │
         ▼
   Predictions
```

Calling `pipeline.fit(train_data)` triggers the entire sequence. The resulting `PipelineModel` can then be applied to new data with a single `transform()` call.

<h2 style="color:#E25822; font-family:Arial, sans-serif;">2. Environment Setup</h2>

We install PySpark and create a `SparkSession`. The cell detects whether we are running on Google Colab and installs dependencies automatically.

In [ ]:
# Install PySpark (only needed on Colab or environments without PySpark)
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install pyspark --quiet
    print("PySpark installed successfully.")
else:
    print("Not running on Colab — assuming PySpark is already installed.")

In [ ]:
# Import core Spark libraries
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, DoubleType

# Create (or retrieve existing) SparkSession
spark = SparkSession.builder \
    .appName("MLlib Machine Learning Demo") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

# Reduce Spark log verbosity
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark version : {spark.version}")
print(f"Spark UI      : {spark.sparkContext.uiWebUrl}")

<h2 style="color:#E25822; font-family:Arial, sans-serif;">Part 1: Regression — Apartment Price Prediction</h2>

We build a regression model to **predict apartment prices** based on:
- `City` — categorical feature (one of five Jordanian cities)
- `Bedrooms` — number of bedrooms
- `Bathrooms` — number of bathrooms
- `Square_Area` — size in square meters

We compare **Linear Regression** and **Decision Tree Regression**.

### 1.1 Generate the Apartment Dataset

We generate 120 synthetic apartment records using Python's `random` module, then load them into a Spark DataFrame.

In [ ]:
import random

random.seed(42)

# City-specific base prices (JOD per m²) — same dataset as Notebook 3
city_base_price = {
    "Amman":  800,
    "Irbid":  500,
    "Zarqa":  400,
    "Aqaba":  600,
    "Madaba": 450,
}
cities = list(city_base_price.keys())

apartment_data = []
for _ in range(120):
    city       = random.choice(cities)
    bedrooms   = random.randint(1, 5)
    bathrooms  = random.randint(1, 3)
    area       = random.randint(45, 250)        # square meters
    noise      = random.gauss(0, 8000)          # price noise
    price      = (city_base_price[city] * area
                  + bedrooms * 3000
                  + bathrooms * 2000
                  + noise)
    price      = max(20000, round(price, -2))   # floor at 20 000 JOD
    apartment_data.append((city, bedrooms, bathrooms, area, float(price)))

# Define schema for the Spark DataFrame
apt_schema = StructType([
    StructField("City",        StringType(),  True),
    StructField("Bedrooms",    IntegerType(), True),
    StructField("Bathrooms",   IntegerType(), True),
    StructField("Square_Area", IntegerType(), True),
    StructField("Price",       DoubleType(),  True),
])

apt_df = spark.createDataFrame(apartment_data, schema=apt_schema)

print(f"Dataset size: {apt_df.count()} rows, {len(apt_df.columns)} columns")
apt_df.show(8, truncate=False)

### 1.2 Exploratory Data Analysis

In [ ]:
# Summary statistics for numerical columns
print("=== Summary Statistics ===")
apt_df.select("Bedrooms", "Bathrooms", "Square_Area", "Price").describe().show()

# Average price per city
print("=== Average Price by City ===")
apt_df.groupBy("City") \
      .agg(F.round(F.avg("Price"), 0).alias("Avg_Price"),
           F.count("*").alias("Count")) \
      .orderBy(F.desc("Avg_Price")) \
      .show()

In [ ]:
# Pearson correlation of numeric features with Price
print("=== Correlation with Price ===")
for col in ["Bedrooms", "Bathrooms", "Square_Area"]:
    corr = apt_df.stat.corr(col, "Price")
    print(f"  {col:15s}  r = {corr:.4f}")

### 1.3 Preprocessing — Encoding Categorical Features

MLlib algorithms require numeric input. We convert the `City` string column using:
1. **`StringIndexer`** — maps each city name to a numeric index (0, 1, 2, …)
2. **`OneHotEncoder`** — converts the index into a sparse binary vector (avoids ordinal bias)

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# Step 1: StringIndexer — City → City_Index
city_indexer = StringIndexer(inputCol="City", outputCol="City_Index")
apt_indexed  = city_indexer.fit(apt_df).transform(apt_df)

# Step 2: OneHotEncoder — City_Index → City_Vec
city_encoder = OneHotEncoder(inputCol="City_Index", outputCol="City_Vec")
apt_encoded  = city_encoder.fit(apt_indexed).transform(apt_indexed)

# Inspect the encoded columns
apt_encoded.select("City", "City_Index", "City_Vec").show(6, truncate=False)

### 1.4 Feature Engineering — VectorAssembler

`VectorAssembler` merges multiple columns into a **single dense/sparse feature vector** — the format expected by all MLlib estimators.

In [ ]:
# Combine the OHE city vector + numeric features into one 'features' column
assembler = VectorAssembler(
    inputCols=["City_Vec", "Bedrooms", "Bathrooms", "Square_Area"],
    outputCol="features"
)

apt_final = assembler.transform(apt_encoded)

# Keep only features and label (Price)
apt_ml = apt_final.select("features", "Price").withColumnRenamed("Price", "label")

print("Feature vector example:")
apt_ml.select("features", "label").show(5, truncate=False)

### 1.5 Train / Test Split

In [ ]:
# 80% training, 20% testing — seed for reproducibility
train_apt, test_apt = apt_ml.randomSplit([0.8, 0.2], seed=42)

print(f"Training set : {train_apt.count()} rows")
print(f"Test set     : {test_apt.count()} rows")

### 1.6 Linear Regression

In [ ]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Instantiate and train Linear Regression
lr = LinearRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.01,     # L2 regularisation strength
    elasticNetParam=0  # 0 = Ridge, 1 = Lasso
)

lr_model  = lr.fit(train_apt)
lr_preds  = lr_model.transform(test_apt)

# Evaluate using RMSE and R²
evaluator_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
evaluator_r2   = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

lr_rmse = evaluator_rmse.evaluate(lr_preds)
lr_r2   = evaluator_r2.evaluate(lr_preds)

print(f"Linear Regression — RMSE : {lr_rmse:,.0f} JOD")
print(f"Linear Regression — R²   : {lr_r2:.4f}")

# Show sample predictions vs actuals
print("\nSample Predictions:")
lr_preds.select(
    F.round("label", 0).alias("Actual_Price"),
    F.round("prediction", 0).alias("Predicted_Price")
).show(8)

### 1.7 Decision Tree Regressor

In [ ]:
from pyspark.ml.regression import DecisionTreeRegressor

# Instantiate and train Decision Tree Regressor
dt = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="label",
    maxDepth=5,        # maximum depth of the tree
    minInstancesPerNode=3
)

dt_model = dt.fit(train_apt)
dt_preds = dt_model.transform(test_apt)

dt_rmse = evaluator_rmse.evaluate(dt_preds)
dt_r2   = evaluator_r2.evaluate(dt_preds)

print(f"Decision Tree Regressor — RMSE : {dt_rmse:,.0f} JOD")
print(f"Decision Tree Regressor — R²   : {dt_r2:.4f}")

In [ ]:
# Feature importance from the Decision Tree
# The feature vector order: [City_Vec(4 dims), Bedrooms, Bathrooms, Square_Area]
feature_names = [f"City_{i}" for i in range(4)] + ["Bedrooms", "Bathrooms", "Square_Area"]
importances   = dt_model.featureImportances

print("Feature Importances (Decision Tree):")
print("-" * 35)
importance_pairs = sorted(
    zip(feature_names, importances.toArray()),
    key=lambda x: -x[1]
)
for name, imp in importance_pairs:
    bar = "█" * int(imp * 50)
    print(f"  {name:12s}  {imp:.4f}  {bar}")

### 1.8 Regression Model Comparison

In [ ]:
print("\n" + "=" * 55)
print(f"{'Model':<30} {'RMSE (JOD)':>12} {'R²':>8}")
print("=" * 55)
print(f"{'Linear Regression':<30} {lr_rmse:>12,.0f} {lr_r2:>8.4f}")
print(f"{'Decision Tree Regressor':<30} {dt_rmse:>12,.0f} {dt_r2:>8.4f}")
print("=" * 55)
print()
winner_rmse = "Linear Regression" if lr_rmse < dt_rmse else "Decision Tree Regressor"
winner_r2   = "Linear Regression" if lr_r2   > dt_r2   else "Decision Tree Regressor"
print(f"Best RMSE : {winner_rmse}")
print(f"Best R²   : {winner_r2}")

<h2 style="color:#E25822; font-family:Arial, sans-serif;">Part 2: Classification — Customer Churn Prediction</h2>

We predict whether a telecom customer will **churn** (cancel their subscription) using:
- `Age` — customer age
- `MonthlyCharges` — monthly bill in EUR
- `Tenure` — months as a customer
- `NumProducts` — number of subscribed products

Target: `Churned` (0 = stayed, 1 = churned)

### 2.1 Generate the Customer Churn Dataset

In [ ]:
import random
import math

random.seed(7)

churn_data = []
for _ in range(220):
    age            = random.randint(18, 70)
    tenure         = random.randint(1, 72)          # months
    num_products   = random.randint(1, 5)
    monthly        = round(random.uniform(20.0, 150.0), 2)

    # Churn probability: higher charges + fewer products + shorter tenure → more likely to churn
    logit  = (-0.03 * tenure
               + 0.02 * monthly
               - 0.3  * num_products
               + 0.5)
    prob   = 1 / (1 + math.exp(-logit))
    churned = 1 if random.random() < prob else 0

    churn_data.append((age, float(monthly), tenure, num_products, churned))

churn_schema = StructType([
    StructField("Age",            IntegerType(), True),
    StructField("MonthlyCharges", DoubleType(),  True),
    StructField("Tenure",         IntegerType(), True),
    StructField("NumProducts",    IntegerType(), True),
    StructField("Churned",        IntegerType(), True),
])

churn_df = spark.createDataFrame(churn_data, schema=churn_schema)

print(f"Dataset size: {churn_df.count()} rows")
churn_df.show(8)

### 2.2 Data Exploration and Class Distribution

In [ ]:
print("=== Summary Statistics ===")
churn_df.describe().show()

print("=== Class Distribution ===")
churn_df.groupBy("Churned") \
        .agg(F.count("*").alias("Count")) \
        .withColumn("Percentage",
                    F.round(F.col("Count") / churn_df.count() * 100, 1)) \
        .orderBy("Churned") \
        .show()

In [ ]:
# Correlation of numeric features with churn label
print("=== Feature Correlations with Churn ===")
for col in ["Age", "MonthlyCharges", "Tenure", "NumProducts"]:
    corr = churn_df.stat.corr(col, "Churned")
    print(f"  {col:18s}  r = {corr:+.4f}")

### 2.3 Feature Engineering and Scaling

In [ ]:
from pyspark.ml.feature import StandardScaler

# Assemble all numeric features into a single vector
churn_assembler = VectorAssembler(
    inputCols=["Age", "MonthlyCharges", "Tenure", "NumProducts"],
    outputCol="raw_features"
)

# StandardScaler: zero mean, unit variance — important for Logistic Regression
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=True,
    withStd=True
)

# Apply transformations
churn_assembled = churn_assembler.transform(churn_df)
scaler_model    = scaler.fit(churn_assembled)
churn_scaled    = scaler_model.transform(churn_assembled)

# Rename label column to 'label' (required by MLlib estimators)
churn_ml = churn_scaled.select("features", F.col("Churned").cast(DoubleType()).alias("label"))

print("Prepared dataset sample:")
churn_ml.show(4, truncate=False)

In [ ]:
# Train / Test Split
train_churn, test_churn = churn_ml.randomSplit([0.8, 0.2], seed=42)
print(f"Training set : {train_churn.count()} rows")
print(f"Test set     : {test_churn.count()} rows")

### 2.4 Logistic Regression

In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Train Logistic Regression
log_reg = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.01
)

lr_cls_model = log_reg.fit(train_churn)
lr_cls_preds = lr_cls_model.transform(test_churn)

# Accuracy
acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
lr_accuracy = acc_evaluator.evaluate(lr_cls_preds)
print(f"Logistic Regression — Accuracy: {lr_accuracy:.4f}  ({lr_accuracy*100:.1f}%)")

# Confusion matrix (manual computation using groupBy)
print("\nConfusion Matrix (Logistic Regression):")
print(f"{'':10s} {'Predicted 0':>14} {'Predicted 1':>14}")
cm = lr_cls_preds.groupBy("label", "prediction").count().collect()
cm_dict = {(int(r["label"]), int(r["prediction"])): r["count"] for r in cm}
for actual in [0, 1]:
    row = f"Actual {actual}   "
    for pred in [0, 1]:
        row += f"{cm_dict.get((actual, pred), 0):>14}"
    print(row)

### 2.5 Random Forest Classifier

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

# Train Random Forest with 50 trees
rf_cls = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=5,
    seed=42
)

rf_cls_model = rf_cls.fit(train_churn)
rf_cls_preds = rf_cls_model.transform(test_churn)

# Accuracy
rf_accuracy = acc_evaluator.evaluate(rf_cls_preds)
print(f"Random Forest Classifier — Accuracy: {rf_accuracy:.4f}  ({rf_accuracy*100:.1f}%)")

# ROC-AUC
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
rf_auc = auc_evaluator.evaluate(rf_cls_preds)
print(f"Random Forest Classifier — ROC-AUC : {rf_auc:.4f}")

In [ ]:
# Classification model comparison
lr_auc = auc_evaluator.evaluate(lr_cls_model.transform(test_churn))

print("\n" + "=" * 60)
print(f"{'Model':<30} {'Accuracy':>10} {'ROC-AUC':>10}")
print("=" * 60)
print(f"{'Logistic Regression':<30} {lr_accuracy:>10.4f} {lr_auc:>10.4f}")
print(f"{'Random Forest Classifier':<30} {rf_accuracy:>10.4f} {rf_auc:>10.4f}")
print("=" * 60)

<h2 style="color:#E25822; font-family:Arial, sans-serif;">Part 3: ML Pipelines</h2>

### What is a Pipeline and Why Use It?

A `Pipeline` encapsulates a sequence of data transformation and model training steps into a **single reusable object**.

**Key benefits:**
- **No data leakage**: preprocessing (e.g., scaling, encoding) is fitted *only* on training data, then applied to test data.
- **Reproducibility**: the exact same transformation sequence is always applied.
- **Deployment simplicity**: save one object; it contains both the transformations and the model.
- **Cross-validation integration**: `CrossValidator` wraps the entire pipeline, ensuring correct train/test separation at every fold.

### Pipeline Structure for This Example

```
StringIndexer  →  OneHotEncoder  →  VectorAssembler  →  StandardScaler  →  RandomForestClassifier
```

We rebuild the apartment churn classification end-to-end using a Pipeline on the raw `churn_df` (before any manual preprocessing).

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Prepare raw churn data with a proper label column (no preprocessing yet)
churn_raw = churn_df.withColumn("label", F.col("Churned").cast(DoubleType()))

# Split the RAW data (the pipeline will handle all preprocessing internally)
train_raw, test_raw = churn_raw.randomSplit([0.8, 0.2], seed=42)

# --- Define Pipeline Stages ---

# Stage 1: Assemble numeric features into a raw vector
pipe_assembler = VectorAssembler(
    inputCols=["Age", "MonthlyCharges", "Tenure", "NumProducts"],
    outputCol="raw_features"
)

# Stage 2: Scale features (fitted on training data only — preventing data leakage)
pipe_scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=True,
    withStd=True
)

# Stage 3: Random Forest Classifier
pipe_rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    seed=42
)

# Build the Pipeline object
pipeline = Pipeline(stages=[pipe_assembler, pipe_scaler, pipe_rf])

print("Pipeline stages:")
for i, stage in enumerate(pipeline.getStages()):
    print(f"  Stage {i}: {type(stage).__name__}")

In [ ]:
# Fit the pipeline on training data (all stages fit/transform in sequence)
pipeline_model = pipeline.fit(train_raw)

# Apply the fitted pipeline to test data
pipeline_preds = pipeline_model.transform(test_raw)

# Evaluate
pipe_accuracy = acc_evaluator.evaluate(pipeline_preds)
pipe_auc      = auc_evaluator.evaluate(pipeline_preds)

print(f"Pipeline Model — Accuracy : {pipe_accuracy:.4f}  ({pipe_accuracy*100:.1f}%)")
print(f"Pipeline Model — ROC-AUC  : {pipe_auc:.4f}")

### 3.1 Saving and Loading a Pipeline Model

A fitted `PipelineModel` can be saved to disk (local filesystem or HDFS/S3) and reloaded later — useful for production deployment.

In [ ]:
import tempfile
import os
from pyspark.ml import PipelineModel

# Use a temporary directory so this works on both Colab and local environments
save_dir = os.path.join(tempfile.gettempdir(), "churn_pipeline_model")

# Save the trained pipeline model
pipeline_model.write().overwrite().save(save_dir)
print(f"Pipeline model saved to: {save_dir}")

# Reload the model from disk
loaded_model = PipelineModel.load(save_dir)
print("Pipeline model loaded successfully.")

# Verify loaded model produces identical predictions
loaded_preds    = loaded_model.transform(test_raw)
loaded_accuracy = acc_evaluator.evaluate(loaded_preds)
print(f"Loaded model accuracy: {loaded_accuracy:.4f}  (should match {pipe_accuracy:.4f})")

<h2 style="color:#E25822; font-family:Arial, sans-serif;">Part 4: Cross-Validation &amp; Hyperparameter Tuning</h2>

Manually choosing hyperparameters is unreliable. **Cross-Validation** with a **parameter grid** automates the search:

1. `ParamGridBuilder` defines the combinations of hyperparameters to evaluate.
2. `CrossValidator` splits the training data into *k* folds, trains on *k-1* folds, validates on the remaining fold, and repeats for each parameter combination.
3. The combination with the best average validation metric is selected as the **best model**.

We tune a `RandomForestClassifier` on the churn dataset using 5-fold cross-validation.

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# Build a fresh pipeline without pre-tuned hyperparameters
cv_assembler = VectorAssembler(
    inputCols=["Age", "MonthlyCharges", "Tenure", "NumProducts"],
    outputCol="raw_features"
)
cv_scaler = StandardScaler(inputCol="raw_features", outputCol="features",
                            withMean=True, withStd=True)
cv_rf     = RandomForestClassifier(featuresCol="features", labelCol="label", seed=42)

cv_pipeline = Pipeline(stages=[cv_assembler, cv_scaler, cv_rf])

# Define the hyperparameter grid
# 2 × 2 × 2 = 8 combinations × 5 folds = 40 model fits
param_grid = (
    ParamGridBuilder()
    .addGrid(cv_rf.numTrees,  [20, 50])      # number of trees
    .addGrid(cv_rf.maxDepth,  [3, 7])        # max tree depth
    .addGrid(cv_rf.minInstancesPerNode, [1, 5])  # minimum samples per leaf
    .build()
)

print(f"Total parameter combinations : {len(param_grid)}")
print(f"Total model fits (5 folds)   : {len(param_grid) * 5}")

# Set up CrossValidator
cross_val = CrossValidator(
    estimator=cv_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=auc_evaluator,
    numFolds=5,
    seed=42
)

print("\nRunning cross-validation (this may take a moment)...")
cv_model = cross_val.fit(train_raw)
print("Cross-validation complete.")

In [ ]:
# Evaluate best model on the held-out test set
best_model   = cv_model.bestModel
cv_preds     = cv_model.transform(test_raw)
cv_accuracy  = acc_evaluator.evaluate(cv_preds)
cv_auc       = auc_evaluator.evaluate(cv_preds)

print(f"Best Model — Test Accuracy : {cv_accuracy:.4f}  ({cv_accuracy*100:.1f}%)")
print(f"Best Model — ROC-AUC       : {cv_auc:.4f}")

# Extract best hyperparameters from the best Random Forest stage
best_rf_stage = best_model.stages[-1]  # last stage is the classifier
print("\nBest Hyperparameters found by CrossValidator:")
print(f"  numTrees               : {best_rf_stage.getNumTrees}")
print(f"  maxDepth               : {best_rf_stage.getMaxDepth()}")
print(f"  minInstancesPerNode    : {best_rf_stage.getMinInstancesPerNode()}")

# Print average metrics per fold combination
print("\nCross-Validation Fold Averages (AUC per parameter combination):")
for i, (params, avg_metric) in enumerate(zip(param_grid, cv_model.avgMetrics)):
    n_trees  = params[cv_rf.numTrees]
    depth    = params[cv_rf.maxDepth]
    min_inst = params[cv_rf.minInstancesPerNode]
    print(f"  [{i+1}] numTrees={n_trees:2d}, maxDepth={depth}, "
          f"minInstances={min_inst}  →  AUC = {avg_metric:.4f}")

<h2 style="color:#E25822; font-family:Arial, sans-serif;">Part 5: Clustering — K-Means</h2>

**K-Means** is an unsupervised algorithm that partitions data into *k* clusters by minimizing the within-cluster sum of squared distances to each cluster centroid.

We generate a 2D dataset with **3 natural clusters** and use K-Means to recover them. We then apply the **Elbow Method** to choose the optimal *k*.

### 5.1 Generate 2D Clustered Dataset

In [ ]:
import random

random.seed(99)

# Three well-separated cluster centres
cluster_centres = [
    (2.0, 3.0),
    (8.0, 8.0),
    (5.0, 1.0),
]

cluster_data = []
for true_label, (cx, cy) in enumerate(cluster_centres):
    for _ in range(60):   # 60 points per cluster → 180 total
        x = cx + random.gauss(0, 0.8)
        y = cy + random.gauss(0, 0.8)
        cluster_data.append((float(x), float(y), true_label))

cluster_schema = StructType([
    StructField("x",          DoubleType(),  True),
    StructField("y",          DoubleType(),  True),
    StructField("true_label", IntegerType(), True),
])

cluster_df = spark.createDataFrame(cluster_data, schema=cluster_schema)

print(f"Cluster dataset: {cluster_df.count()} points across 3 clusters")
cluster_df.show(6)

### 5.2 K-Means with k=3

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Assemble x, y into a feature vector
kmeans_assembler = VectorAssembler(inputCols=["x", "y"], outputCol="features")
cluster_features = kmeans_assembler.transform(cluster_df)

# Fit K-Means with k=3
kmeans = KMeans(
    featuresCol="features",
    predictionCol="cluster",
    k=3,
    seed=42,
    maxIter=20
)
km_model    = kmeans.fit(cluster_features)
km_result   = km_model.transform(cluster_features)

# Cluster centres
print("Cluster Centres (K-Means, k=3):")
for i, centre in enumerate(km_model.clusterCenters()):
    print(f"  Cluster {i}: x={centre[0]:.3f}, y={centre[1]:.3f}")

# Silhouette score: ranges from -1 (bad) to +1 (perfect separation)
sil_evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="cluster",
    metricName="silhouette"
)
sil_score = sil_evaluator.evaluate(km_result)
print(f"\nSilhouette Score (k=3): {sil_score:.4f}")

# Cluster assignment distribution
print("\nPoints per cluster:")
km_result.groupBy("cluster").count().orderBy("cluster").show()

### 5.3 Elbow Method — Choosing the Optimal k

The **Elbow Method** tries multiple values of *k* and plots the **Within-Cluster Sum of Squared Errors (WCSSE)** (also called inertia). We look for the "elbow" — the point where adding more clusters yields diminishing returns.

In [ ]:
# Evaluate K-Means for k = 2 to 7 and record WCSSE and Silhouette score
elbow_results = []

for k in range(2, 8):
    km_k      = KMeans(featuresCol="features", predictionCol="cluster",
                       k=k, seed=42, maxIter=20)
    model_k   = km_k.fit(cluster_features)
    preds_k   = model_k.transform(cluster_features)
    wcsse     = model_k.summary.trainingCost   # sum of squared distances to centroid
    sil_k     = sil_evaluator.evaluate(preds_k)
    elbow_results.append((k, round(wcsse, 2), round(sil_k, 4)))
    print(f"  k={k}  WCSSE={wcsse:>10,.2f}  Silhouette={sil_k:.4f}")

print("\nDone.")

In [ ]:
# ASCII bar chart for visual inspection of the Elbow Method
print("\nElbow Method — WCSSE by k")
print("=" * 55)
max_wcsse = max(r[1] for r in elbow_results)
for k, wcsse, sil in elbow_results:
    bar_len = int((wcsse / max_wcsse) * 35)
    bar     = "█" * bar_len
    marker  = " ← elbow" if k == 3 else ""
    print(f"  k={k}  {bar:<36} {wcsse:>10,.0f}{marker}")
print("=" * 55)

### Elbow Method Results Summary

| k | WCSSE (lower = better fit) | Silhouette Score (higher = better separation) |
|---|---|---|
| 2 | High | Low–Medium |
| 3 | **Elbow point — large drop from k=2** | **Highest** |
| 4 | Moderate decrease | Decreasing |
| 5 | Small decrease | Further decrease |
| 6 | Marginal decrease | Low |
| 7 | Marginal decrease | Low |

The elbow at **k=3** corresponds to our 3 natural clusters, confirming the method works correctly on this dataset.

<h2 style="color:#E25822; font-family:Arial, sans-serif;">Summary — Algorithms Covered</h2>

| Algorithm | Type | MLlib Class | Key Hyperparameters | When to Use |
|---|---|---|---|---|
| **Linear Regression** | Regression | `LinearRegression` | `maxIter`, `regParam`, `elasticNetParam` | Continuous target, features have roughly linear relationship with target |
| **Decision Tree Regressor** | Regression | `DecisionTreeRegressor` | `maxDepth`, `minInstancesPerNode` | Non-linear relationships; need interpretable feature importances |
| **Logistic Regression** | Classification | `LogisticRegression` | `maxIter`, `regParam` | Binary or multiclass classification; fast, interpretable, good baseline |
| **Random Forest Classifier** | Classification | `RandomForestClassifier` | `numTrees`, `maxDepth`, `featureSubsetStrategy` | Robust, handles missing values, provides feature importance, reduces overfitting vs single tree |
| **K-Means** | Clustering | `KMeans` | `k`, `maxIter`, `tol` | Discover natural groupings in unlabelled data; works best with spherical clusters |
| **Pipeline** | Workflow | `Pipeline` | — | Combine preprocessing + model into a single reusable, leak-free object |
| **CrossValidator** | Tuning | `CrossValidator` | `numFolds` | Automated hyperparameter search with honest performance estimation |

### Key Takeaways

- Always use **Pipelines** in production — they prevent data leakage and simplify deployment.
- **StandardScaler** is essential for distance-based and gradient-based algorithms (Logistic Regression, K-Means) but not needed for tree-based methods.
- **Cross-Validation** gives a more honest estimate of model performance than a single train/test split.
- For classification, use **ROC-AUC** alongside accuracy — accuracy can be misleading on imbalanced datasets.
- The **Elbow Method** and **Silhouette Score** together guide the choice of *k* in K-Means.

<h2 style="color:#E25822; font-family:Arial, sans-serif;">Exercises</h2>

Work through the following exercises to reinforce your understanding of Spark MLlib.

---

### Exercise 1 — Gradient Boosted Tree Regressor

The `GBTRegressor` often outperforms both Linear Regression and single Decision Trees on tabular data.

1. Import `GBTRegressor` from `pyspark.ml.regression`.
2. Train a `GBTRegressor` on the `train_apt` dataset using `maxIter=50` and `maxDepth=4`.
3. Evaluate RMSE and R² on `test_apt`.
4. Add the GBT results to the regression comparison table from Part 1.
5. **Bonus**: Use `CrossValidator` with a `ParamGridBuilder` to search over `maxDepth` ∈ {3, 5, 7} and `maxIter` ∈ {30, 50}. Report the best parameters.

---

### Exercise 2 — Extend the Churn Pipeline with a Categorical Feature

The current churn dataset only has numeric features. Extend it with a categorical column.

1. Add a `Contract` column to `churn_data` with values `"Monthly"`, `"Yearly"`, `"Two-Year"` (assigned randomly). Assign higher churn probability to `"Monthly"` contracts.
2. Rebuild the Pipeline from Part 3 to include:
   - `StringIndexer` for the `Contract` column
   - `OneHotEncoder` for the indexed column
   - `VectorAssembler` merging the OHE vector with the other numeric features
   - `StandardScaler` (only on the numeric features, not the OHE vector — or explain why scaling OHE is unnecessary)
   - `RandomForestClassifier`
3. Compare the accuracy and ROC-AUC with the pipeline from Part 3.

---

### Exercise 3 — Customer Segmentation with K-Means

Apply K-Means clustering to the **churn dataset** to segment customers into behavioural groups.

1. Use the scaled churn features (`churn_scaled`) from Part 2.
2. Run the Elbow Method for *k* = 2 to 8. Plot the WCSSE values in a formatted table or ASCII chart.
3. Select the best *k* based on the Elbow Method and Silhouette Score.
4. For each cluster, compute the average `MonthlyCharges`, `Tenure`, `NumProducts`, and `Churned` rate. Describe in plain language what each cluster represents (e.g., *"low-value recent customers with high churn risk"*).
5. **Bonus**: Discuss whether the cluster memberships align with the churn predictions from the Random Forest Classifier in Part 2.

In [ ]:
# --- Your workspace for the exercises ---
# Exercise 1: GBT Regressor

# Exercise 2: Extended Churn Pipeline

# Exercise 3: Customer Segmentation


In [ ]:
# Cleanly stop the SparkSession when done
spark.stop()
print("SparkSession stopped.")